# Notebook ٩: Transformers وتوليد النص (GPT-2 الحقيقي)

Attention بإيدينا وبـ Keras، ثم GPT-2 حقيقي: احتمالات الكلمة الجاية وtemperature وtop-k/top-p وperplexity. كل خلية اتشغلت فعلًا.

## ١. Attention بإيدينا

كل كلمة بتبص على الكلمات اللي قبلها وتاخد منها معلومات بأوزان.

In [ ]:
# المصفوفات
import numpy as np
# seed ثابت
np.random.seed(0)
# 4 كلمات
words = ["salah", "scored", "a", "goal"]
# بعد المتجه
d = 8
# متجه عشوائي لكل كلمة (للشرح بس)
X = np.random.randn(4, d)
# مصفوفات query وkey وvalue
Wq, Wk, Wv = [np.random.randn(d, d) for _ in range(3)]
# نحسب Q وK وV
Q, K, V = X @ Wq, X @ Wk, X @ Wv
# درجة التشابه مقسومة على جذر البعد
scores = Q @ K.T / np.sqrt(d)
# مصفوفة مثلثية: الكلمة ما تشوفش المستقبل
mask = np.tril(np.ones((4, 4)))
# نخفي المستقبل بـ -inf
scores = np.where(mask == 1, scores, -np.inf)
# بسط softmax
w = np.exp(scores - scores.max(1, keepdims=True))
# الأوزان بعد softmax
w = w / w.sum(1, keepdims=True)
# صفوف = الكلمة الحالية، أعمدة = الكلمة اللي بتبص عليها
print(np.round(w, 2))
# الناتج: مجموع موزون للـ values
out = w @ V
# شكل الناتج
print(out.shape)


## ٢. نفس الفكرة بـ Keras

طبقة MultiHeadAttention جاهزة مع causal mask.

In [ ]:
# نخفي رسايل TensorFlow
import os; os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
# TensorFlow
import tensorflow as tf
# طبقة attention بـ 2 heads
mha = tf.keras.layers.MultiHeadAttention(num_heads=2, key_dim=4)
# نفس X بـ batch واحد
x = tf.constant(X[None], dtype=tf.float32)
# self-attention مع قناع سببي
y, att = mha(x, x, use_causal_mask=True, return_attention_scores=True)
# شكل الناتج وشكل الأوزان (batch, heads, من، إلى)
print(y.shape, att.shape)
# أوزان head الأول
print(np.round(att.numpy()[0, 0], 2))


## ٣. GPT-2 الحقيقي: الكلمة الجاية

الموديل بيطلع احتمال لكل توكن في الـ vocabulary (50257).

In [ ]:
# نخفي التحذيرات
import warnings; warnings.filterwarnings("ignore")
# backend موديلات Hugging Face
import torch
# الـ tokenizer والموديل
from transformers import AutoTokenizer, AutoModelForCausalLM
# tokenizer GPT-2
tok = AutoTokenizer.from_pretrained("gpt2")
# موديل GPT-2 في وضع الاستخدام
lm = AutoModelForCausalLM.from_pretrained("gpt2").eval()
# عدد الأوزان بالملايين
print(round(sum(p.numel() for p in lm.parameters()) / 1e6), "M parameters")
# دالة بترجع احتمالات الكلمة الجاية
def next_probs(prompt):
# نحول النص لأرقام
    ids = tok(prompt, return_tensors="pt")
# من غير حساب gradients
    with torch.no_grad():
# logits آخر موضع
        logits = lm(**ids).logits[0, -1]
# نرجعها
    return logits
# نجرب جملة
logits = next_probs("Mohamed Salah is a famous football")
# نحول logits لاحتمالات
p = torch.softmax(logits, -1)
# أعلى 5
top = p.topk(5)
# الكلمة واحتمالها
print([(tok.decode(int(i)), round(float(v), 3)) for v, i in zip(top.values, top.indices)])


## ٤. Temperature وtop-k وtop-p

نفس الـ logits بنغير طريقة الاختيار منها.

In [ ]:
# دالة تعرض أعلى k
def show(probs, k=4):
# أعلى k
    t = probs.topk(k)
# كلمات واحتمالات
    return [(tok.decode(int(i)).strip(), round(float(v), 2)) for v, i in zip(t.values, t.indices)]
# 3 قيم temperature
for T in [0.5, 1.0, 2.0]:
# نقسم logits على T ثم softmax
    print("T =", T, show(torch.softmax(logits / T, -1)))


In [ ]:
# نبقي أعلى k بس
def top_k_probs(logits, k):
# أعلى k logits
    v, i = logits.topk(k)
# الباقي -inf
    out = torch.zeros_like(logits).fill_(-float("inf"))
# نرجع القيم الباقية
    out[i] = v
# نعيد التطبيع
    return torch.softmax(out, -1)
# nucleus: أصغر مجموعة مجموعها ≥ p
def top_p_probs(logits, p):
# احتمالات
    pr = torch.softmax(logits, -1)
# ترتيب تنازلي
    sp, si = pr.sort(descending=True)
# نبقي لحد ما نعدي p
    keep = (sp.cumsum(0) - sp) < p
# احتمالات جديدة
    out = torch.zeros_like(pr)
# نبقي المختارة
    out[si[keep]] = sp[keep]
# نعيد التطبيع
    return out / out.sum()
# top-k بـ k=3
print("top-3:", show(top_k_probs(logits, 3), 3))
# top-p بـ 0.5
pp = top_p_probs(logits, 0.5)
# عدد التوكنز المتبقية
print("top-p 0.5 keeps", int((pp > 0).sum()), "tokens:", show(pp, 4))


## ٥. توليد نص

autoregressive: نختار توكن، نضيفه، ونكرر.

In [ ]:
# seed ثابت
torch.manual_seed(0)
# بداية النص
ids = tok("The goalkeeper made a", return_tensors="pt")["input_ids"]
# 12 توكن
for _ in range(12):
# من غير gradients
    with torch.no_grad():
# logits آخر موضع
        lg = lm(input_ids=ids).logits[0, -1]
# temperature 0.8 ثم top-p 0.9
    pr = top_p_probs(lg / 0.8, 0.9)
# نسحب توكن عشوائي حسب الاحتمالات
    nxt = torch.multinomial(pr, 1)
# نضيفه للنص
    ids = torch.cat([ids, nxt[None]], dim=1)
# النص الناتج
print(tok.decode(ids[0]))


## ٦. Perplexity

مقياس لقد إيه الموديل مستغرب من النص: أقل = أفضل.

In [ ]:
# perplexity لنص
def ppl(text):
# نحول النص لأرقام
    ids = tok(text, return_tensors="pt")["input_ids"]
# من غير gradients
    with torch.no_grad():
# متوسط cross-entropy
        loss = lm(input_ids=ids, labels=ids).loss
# perplexity = exp(loss)
    return round(float(torch.exp(loss)), 1)
# جملة طبيعية
print(ppl("The striker scored a goal in the final minute of the match."))
# نفس الكلمات بترتيب عشوائي
print(ppl("Match the of minute final goal striker a scored the in."))
